# Dark Current and C-V Evolution Under Proton Irradiation

This notebook combines two key radiation damage observables for a 10 $\mu$m 4H-SiC
epitaxial detector under 62 MeV proton irradiation:

1. **Dark current vs fluence** with component decomposition (SRH, TAT, SRV) --
   demonstrates generation lifetime degradation from defect introduction.
2. **C-V evolution** showing progressive flattening of capacitance-voltage
   characteristics as carrier removal reduces the effective doping.
3. **Mott-Schottky ($1/C^2$ vs $V$)** analysis confirming slope decrease (lower
   effective doping) with increasing fluence.
4. **Effective doping profiles** showing position-dependent carrier removal, where
   the weakly doped bulk epi compensates first.

The critical fluence $\Phi_{\mathrm{crit}}$ -- at which the minimum doping point
reaches full compensation -- is computed and annotated throughout.

**Reference:** Burin et al., arXiv:2407.16710 (2024) -- damage constants for 4H-SiC

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from src.dark_current import dark_current_vs_fluence, plot_dark_current_vs_fluence
from src.cv_analysis import cv_at_fluence, plot_cv_evolution
from src.radiation_damage import (
    compute_phi_crit,
    compute_damaged_params,
    RadiationDamageParams,
)
from src.sic_material import srh_lifetime

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'figure.figsize': (8, 6),
})

print('Notebook 11: Dark Current and C-V Evolution Under Proton Irradiation')
print('Device: 10 um 4H-SiC epitaxial detector')
print('Beam: 62 MeV protons')

## Parameters

In [ ]:
# Dark current fluence range: 30 points from 1e10 to 1e14, with 0 prepended
fluence_range = np.concatenate([[0.0], np.geomspace(1e10, 1e14, 30)])

# C-V fluence levels: selected to show progression without exceeding Phi_crit
fluence_levels = [0, 1e12, 5e12, 1e13, 5e13]

# Voltage range for C-V sweeps: 0V to -30V in ~2V steps
V_range = np.linspace(0, -30, 16)

# Dark current bias
V_bias = -30.0

# Proton energy
energy_MeV = 62.0

# Compute Phi_crit using an approximate graded epi profile
# (actual cv_at_fluence calls use the real device profile internally)
N_D_approx = np.linspace(8.5e13, 2.9e15, 50)
phi_crit_info = compute_phi_crit(N_D_approx, eta_removal=5.0, energy_MeV=energy_MeV)

phi_crit = phi_crit_info['phi_crit_proton']
print(f"Phi_crit = {phi_crit:.3e} protons/cm^2 (62 MeV)")
print(f"  N_D_min = {phi_crit_info['N_D_min']:.3e} cm^-3")
print(f"  kappa   = {phi_crit_info['kappa']:.3f}")
print()
print(f"Dark current: {len(fluence_range)} fluence points, V_bias = {V_bias} V")
print(f"C-V: {len(fluence_levels)} fluence levels, V_range = [{V_range[0]:.0f}, {V_range[-1]:.0f}] V")

## Panel (a): Dark Current vs Fluence

In [ ]:
# Compute dark current vs fluence at -30V
dc_result = dark_current_vs_fluence(
    fluence_range, V_bias=V_bias, area=0.04, energy_MeV=energy_MeV
)
print(f"Dark current computed at {len(fluence_range)} fluence points")
print(f"Baseline (pristine): {abs(dc_result['I_baseline'])*1e12:.2f} pA")

In [ ]:
# Plot dark current with component decomposition
fig, ax = plt.subplots(figsize=(8, 6))
plot_dark_current_vs_fluence(dc_result, ax=ax)

# Annotate Phi_crit
ax.axvline(phi_crit, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax.annotate(
    f'$\\Phi_{{\\mathrm{{crit}}}}$ = {phi_crit:.2e}',
    xy=(phi_crit, ax.get_ylim()[1] * 0.3),
    xytext=(phi_crit * 0.15, ax.get_ylim()[1] * 0.5),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.2),
    fontsize=10, color='red',
)
ax.set_title('Dark Current vs Proton Fluence (62 MeV, $V = -30$ V)')

fig.tight_layout()
fig.savefig('../figures/11a_dark_current_vs_fluence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/11a_dark_current_vs_fluence.png')

## Panel (b): C-V Evolution Under Irradiation

In [ ]:
# Compute C-V at each fluence level
cv_results = []
for phi in fluence_levels:
    print(f"Computing C-V at fluence = {phi:.1e} p/cm^2 ...")
    result = cv_at_fluence(
        fluence=float(phi),
        V_range=V_range,
        area=1.0,
        energy_MeV=energy_MeV,
    )
    cv_results.append(result)
    if result is None:
        print(f"  -> None (fluence >= Phi_crit)")
    else:
        C_max = np.max(result['capacitance'])
        C_min = np.min(result['capacitance'])
        print(f"  -> C range: [{C_min:.3e}, {C_max:.3e}] F/cm^2")

print(f"\nPhi_crit = {phi_crit:.3e} protons/cm^2")
for phi in fluence_levels:
    ratio = phi / phi_crit if phi > 0 else 0
    flag = ' ** NEAR Phi_crit **' if ratio > 0.8 else ''
    print(f"  {phi:.1e}: {ratio:.1%} of Phi_crit{flag}")

In [ ]:
# Plot C-V evolution: C vs V (left) and 1/C^2 vs V (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: C vs V using plot_cv_evolution
plot_cv_evolution(cv_results, fluence_levels, ax=ax1,
                  title='C-V Curves at Multiple Fluences')

# Right panel: 1/C^2 vs V (Mott-Schottky)
cmap = plt.cm.viridis
valid_fluences = [f for f, r in zip(fluence_levels, cv_results) if r is not None]
f_min = min(f for f in valid_fluences if f > 0) if any(f > 0 for f in valid_fluences) else 1.0
f_max = max(valid_fluences)
norm = plt.Normalize(vmin=0, vmax=f_max)

for phi, cv_r in zip(fluence_levels, cv_results):
    if cv_r is None:
        continue
    color = cmap(norm(phi))
    label = 'Pristine' if phi == 0 else f'{phi:.1e} p/cm$^2$'
    C = np.asarray(cv_r['capacitance'])
    V = np.asarray(cv_r['voltages'])
    # Avoid division by zero
    mask = C > 0
    inv_C2 = np.full_like(C, np.nan)
    inv_C2[mask] = 1.0 / C[mask]**2
    ax2.plot(V, inv_C2, color=color, label=label)

ax2.set_xlabel('Voltage (V)')
ax2.set_ylabel('$1/C^2$ (cm$^4$/F$^2$)')
ax2.set_title('Mott-Schottky: $1/C^2$ vs $V$')
ax2.legend(fontsize=8)

# Add text box with Phi_crit
textstr = f'$\\Phi_{{\\mathrm{{crit}}}}$ = {phi_crit:.2e} p/cm$^2$'
ax2.text(0.02, 0.98, textstr, transform=ax2.transAxes, fontsize=9,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.tight_layout()
fig.savefig('../figures/11b_cv_evolution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/11b_cv_evolution.png')

## Panel (c): Effective Doping Profile at Selected Fluences

In [ ]:
# Visualize position-dependent carrier removal
# Create a graded doping profile (100 nodes) matching the device
N_nodes = 100
N_D_pristine = np.linspace(8.5e13, 2.9e15, N_nodes)
positions_um = np.linspace(0, 10, N_nodes)  # 0 = bulk, 10 um = junction

# Get pristine lifetimes for compute_damaged_params
tau_n_0 = srh_lifetime(300.0, 'electron')
tau_p_0 = srh_lifetime(300.0, 'hole')

# Fluence levels for doping profile visualization
profile_fluences = [0, 1e12, 1e13, 3e13]
profile_labels = [
    'Pristine',
    '$10^{12}$ p/cm$^2$',
    '$10^{13}$ p/cm$^2$',
    '$3{\\times}10^{13}$ p/cm$^2$',
]
profile_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']

fig, ax = plt.subplots(figsize=(8, 6))

for phi, label, color in zip(profile_fluences, profile_labels, profile_colors):
    damaged = compute_damaged_params(
        tau_n_0, tau_p_0, N_D_pristine,
        fluence=float(phi), energy_MeV=energy_MeV,
    )
    N_D_dam = np.maximum(damaged['N_D_profile'], 0)  # clip negatives to zero
    ax.semilogy(positions_um, np.maximum(N_D_dam, 1e10), color=color,
                linewidth=2, label=label)
    # Mark where compensation occurs (N_D drops to ~0)
    comp_mask = damaged['N_D_profile'] <= 0
    if np.any(comp_mask):
        x_comp = positions_um[comp_mask][0]
        ax.axvline(x_comp, color=color, linestyle=':', alpha=0.5)
        ax.annotate(f'Compensation\nat {x_comp:.1f} $\\mu$m',
                    xy=(x_comp, 1e12), fontsize=8, color=color,
                    ha='left', va='bottom')

ax.set_xlabel('Position ($\\mu$m) [0 = bulk, 10 = junction]')
ax.set_ylabel('Effective $N_D$ (cm$^{-3}$)')
ax.set_title('Effective Doping Profile Under Carrier Removal')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(1e10, 1e16)

# Annotate bulk and junction regions
ax.text(1.0, 2e14, 'Bulk epi\n$N_D = 8.5{\\times}10^{13}$', fontsize=9, ha='center')
ax.text(9.0, 1e15, 'Junction\n$N_D = 2.9{\\times}10^{15}$', fontsize=9, ha='center')

fig.tight_layout()
fig.savefig('../figures/11c_doping_profiles.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/11c_doping_profiles.png')

## Combined Figure

In [ ]:
# Create 2x2 combined figure for publication
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- (0,0): Dark current vs fluence ---
ax = axes[0, 0]
plot_dark_current_vs_fluence(dc_result, ax=ax)
ax.axvline(phi_crit, color='red', linestyle=':', linewidth=1.2, alpha=0.7)
ax.annotate(
    f'$\\Phi_{{\\mathrm{{crit}}}}$',
    xy=(phi_crit, ax.get_ylim()[1] * 0.3),
    xytext=(phi_crit * 0.12, ax.get_ylim()[1] * 0.5),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.0),
    fontsize=10, color='red',
)
ax.set_title('(a) Dark Current vs Fluence')

# --- (0,1): C-V curves ---
ax = axes[0, 1]
plot_cv_evolution(cv_results, fluence_levels, ax=ax,
                  title='(b) C-V Evolution with Fluence')

# --- (1,0): Mott-Schottky 1/C^2 vs V ---
ax = axes[1, 0]
for phi, cv_r in zip(fluence_levels, cv_results):
    if cv_r is None:
        continue
    color = cmap(norm(phi))
    label = 'Pristine' if phi == 0 else f'{phi:.1e} p/cm$^2$'
    C = np.asarray(cv_r['capacitance'])
    V = np.asarray(cv_r['voltages'])
    mask = C > 0
    inv_C2 = np.full_like(C, np.nan)
    inv_C2[mask] = 1.0 / C[mask]**2
    ax.plot(V, inv_C2, color=color, label=label)
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('$1/C^2$ (cm$^4$/F$^2$)')
ax.set_title('(c) Mott-Schottky: $1/C^2$ vs $V$')
ax.legend(fontsize=7)

# --- (1,1): Doping profiles ---
ax = axes[1, 1]
for phi, label, color in zip(profile_fluences, profile_labels, profile_colors):
    damaged = compute_damaged_params(
        tau_n_0, tau_p_0, N_D_pristine,
        fluence=float(phi), energy_MeV=energy_MeV,
    )
    N_D_dam = np.maximum(damaged['N_D_profile'], 0)
    ax.semilogy(positions_um, np.maximum(N_D_dam, 1e10), color=color,
                linewidth=1.8, label=label)
ax.set_xlabel('Position ($\\mu$m)')
ax.set_ylabel('Effective $N_D$ (cm$^{-3}$)')
ax.set_title('(d) Doping Profiles Under Carrier Removal')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(1e10, 1e16)

fig.suptitle(
    'Dark Current and C-V Evolution Under 62 MeV Proton Irradiation\n'
    '10 $\\mu$m 4H-SiC Epitaxial Detector',
    fontsize=14, y=1.01,
)
fig.tight_layout()
fig.savefig('../figures/11_dark_current_cv_evolution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/11_dark_current_cv_evolution.png')

## Summary and Observations

### Dark Current (Panel a)

Dark current increases monotonically with proton fluence due to SRH generation
lifetime degradation from defect introduction ($Z_{1/2}$, $EH_{6,7}$). The component
decomposition shows:
- **SRH bulk generation** dominates at moderate fluences
- **TAT (trap-assisted tunneling)** becomes significant at high fields
- **Surface recombination** provides a fluence-independent baseline

### C-V Evolution (Panel b)

C-V curves flatten progressively as carrier removal reduces the effective doping
concentration. At the highest fluences approaching $\Phi_{\mathrm{crit}}$, the
capacitance variation with voltage diminishes significantly, indicating a nearly
fully compensated (intrinsic-like) epitaxial layer.

### Mott-Schottky Analysis (Panel c)

The $1/C^2$ vs $V$ plots confirm doping reduction:
- **Pristine:** steep slope $\propto 1/N_D$
- **High fluence:** shallower slope indicating lower effective doping
- Slope decrease is directly proportional to carrier removal

### Doping Profiles (Panel d)

Position-dependent carrier removal reveals the graded profile's vulnerability:
- **Bulk epi** ($N_D = 8.5 \times 10^{13}$ cm$^{-3}$) compensates first
- **Junction region** ($N_D = 2.9 \times 10^{15}$ cm$^{-3}$) compensates last
- $\Phi_{\mathrm{crit}}$ is determined by the weakest (minimum doping) point

### Critical Fluence

For this device at 62 MeV:
- $\Phi_{\mathrm{crit}} \approx 4.86 \times 10^{13}$ protons/cm$^2$
- Set by bulk epi doping $N_{D,\mathrm{bulk}} = 8.5 \times 10^{13}$ cm$^{-3}$ and
  carrier removal rate $\eta_{\mathrm{removal}} = 5.0$ cm$^{-1}$